# Feature Engineering: Works Dataset (Pandas)

This notebook performs data quality checks and feature engineering on the `works.parquet` dataset using **Pandas**.

## Schema (Input)
- **work_key** (String, nullable)
- **title** (String, nullable)
- **author_key** (String, nullable)
- **subjects** (List[String], nullable)

## Objectives
1. Assess data quality (null values, title/author_key/subjects)
2. Validate `work_key` (required; must start with `/works/`)
3. Validate `author_key` (if present, must start with `/authors/` or be null)
4. Clean `title` (trim whitespace; remove rows with null/empty title)
5. Clean `subjects` (trim list items, empty list → null, limit to first 5)
6. Generate quality report

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 1: Load and Inspect Raw Data

In [2]:
# Load works data (works.parquet only, not works_part1 etc.)
input_path = PROCESSED_DIR / 'works.parquet'
df = pd.read_parquet(input_path)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSchema:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 40,688,965
Columns: ['work_key', 'title', 'author_key', 'subjects']

Schema:
work_key         str
title            str
author_key       str
subjects      object
dtype: object

Memory usage: 8433.53 MB


In [3]:
# Display first few rows
print(df.head(10))
print(df.tail(5))

             work_key                                              title  \
0  /works/OL10000358W                                     Chacun et Tous   
1  /works/OL10000512W                                      PCEM intensif   
2   /works/OL1000092W                                  Highland princess   
3   /works/OL1000101W                                     Wedding belles   
4  /works/OL10001112W                             Connexions dangereuses   
5  /works/OL10001436W                              The revolutionary war   
6  /works/OL10001472W                          Contes du sud du Cameroun   
7  /works/OL10001494W  The catalogue of allmost all the works done by...   
8  /works/OL10001543W                        Vision d'escaflowne, tome 5   
9  /works/OL10001700W                           Cris du corps, nouvelles   

            author_key                                           subjects  
0  /authors/OL3965379A                                                 []  
1  /authors

## Step 2: Data Quality Assessment

In [4]:
# Check null values
print("=== Null Value Counts ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

quality_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage': null_pct.values
})
print(quality_df.to_string(index=False))

=== Null Value Counts ===
    Column  Null Count  Null Percentage
  work_key           0         0.000000
     title          63         0.000155
author_key     2190957         5.384647
  subjects           0         0.000000


In [5]:
# Title: null vs empty string
print("=== Title Analysis ===")
title_null = df['title'].isna().sum()
title_empty = (df['title'].astype(str).str.strip() == '').sum()
title_ok = ((df['title'].notna()) & (df['title'].astype(str).str.strip() != '')).sum()
print(f"Null title: {title_null:,}")
print(f"Empty/whitespace-only title: {title_empty:,}")
print(f"Non-empty title: {title_ok:,}")
print(f"\nSample titles:")
print(df['title'].dropna().head(10).tolist())

=== Title Analysis ===
Null title: 63
Empty/whitespace-only title: 0
Non-empty title: 40,688,902

Sample titles:
['Chacun et Tous', 'PCEM intensif', 'Highland princess', 'Wedding belles', 'Connexions dangereuses', 'The revolutionary war', 'Contes du sud du Cameroun', 'The catalogue of allmost all the works done by Tomoko Takahashi (between 1985-2002)', "Vision d'escaflowne, tome 5", 'Cris du corps, nouvelles']


In [6]:
# work_key validation: required, should start with /works/
print("=== work_key Validation ===")
work_key_non_null = df['work_key'].dropna()
valid_work_key = work_key_non_null.astype(str).str.startswith('/works/', na=False)
print(f"Rows with non-null work_key: {len(work_key_non_null):,}")
print(f"Rows with work_key starting with /works/: {valid_work_key.sum():,}")
print(f"Rows with null or malformed work_key: {len(df) - valid_work_key.sum():,}")

=== work_key Validation ===
Rows with non-null work_key: 40,688,965
Rows with work_key starting with /works/: 40,688,965
Rows with null or malformed work_key: 0


In [7]:
# author_key validation: if present, should start with /authors/
print("=== author_key Validation ===")
author_non_null = df['author_key'].dropna()
author_empty = (df['author_key'].astype(str).str.strip() == '').sum()
valid_author = author_non_null.astype(str).str.startswith('/authors/', na=False)
print(f"Rows with non-null author_key: {len(author_non_null):,}")
print(f"Rows with author_key starting with /authors/: {valid_author.sum():,}")
malformed = author_non_null[~valid_author]
if len(malformed) > 0:
    print(f"Rows with malformed author_key: {len(malformed):,}")
    for v in malformed.head(5):
        print(f"  '{v}'")

=== author_key Validation ===
Rows with non-null author_key: 38,498,008
Rows with author_key starting with /authors/: 38,458,539
Rows with malformed author_key: 39,469
  '{"type":{"key":"/type/author_role"}}'
  '{"type":{"key":"/type/author_role"}}'
  '{"type":{"key":"/type/author_role"}}'
  '{"type":{"key":"/type/author_role"}}'
  '{"type":{"key":"/type/author_role"}}'


In [8]:
# subjects: check structure (list type, lengths)
print("=== subjects Analysis ===")
subjects_null = df['subjects'].isna().sum()
print(f"Null subjects: {subjects_null:,}")
if 'subjects' in df.columns:
    s = df['subjects'].dropna()
    if len(s) > 0:
        lens = s.apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
        print(f"Non-null subjects count: {len(s):,}")
        print(f"Subject list length - min: {lens.min()}, max: {lens.max()}, mean: {lens.mean():.2f}")
        print(f"Sample subject lists: {s.head(3).tolist()}")

=== subjects Analysis ===
Null subjects: 0
Non-null subjects count: 40,688,965
Subject list length - min: 0, max: 0, mean: 0.00
Sample subject lists: [array([], dtype=object), array([], dtype=object), array(['Historical romance', 'Historical fiction', 'Fiction',
       'Fiction, romance, historical, general', 'Scotland, fiction'],
      dtype=object)]


## Step 3: Clean and Transform Data

In [9]:
df_cleaned = df.copy()
print(f"Original row count: {len(df_cleaned):,}")

Original row count: 40,688,965


In [10]:
# Keep only rows with valid work_key (required)
before = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['work_key'].notna() &
    (df_cleaned['work_key'].astype(str).str.strip() != '') &
    df_cleaned['work_key'].astype(str).str.startswith('/works/', na=False)
].copy()
print(f"Rows removed (null/invalid work_key): {before - len(df_cleaned):,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows removed (null/invalid work_key): 0
Rows retained: 40,688,965


In [11]:
# Clean title: strip whitespace, then remove null/empty title rows
df_cleaned['title'] = df_cleaned['title'].astype(str).str.strip()
before = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['title'].notna() & (df_cleaned['title'] != '')
].copy()
print(f"Rows removed (null/empty title): {before - len(df_cleaned):,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows removed (null/empty title): 63
Rows retained: 40,688,902


In [12]:
# Clean author_key: strip; invalid format -> null (keep row)
def clean_author_key(v):
    if pd.isna(v) or v == '': return np.nan
    s = str(v).strip()
    if s == '' or not s.startswith('/authors/'): return np.nan
    return s

df_cleaned['author_key'] = df_cleaned['author_key'].apply(clean_author_key)
print("author_key cleaned: invalid or empty set to null.")

author_key cleaned: invalid or empty set to null.


In [14]:
# Clean subjects: ensure list, strip strings, empty list -> null, limit to first 5
def clean_subjects(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return np.nan
    if isinstance(x, np.ndarray): x = x.tolist()
    if not isinstance(x, (list, tuple)): return np.nan
    items = [str(i).strip() for i in x if str(i).strip()]
    if not items: return np.nan
    return items[:5]  # limit to first 5

df_cleaned['subjects'] = df_cleaned['subjects'].apply(clean_subjects)
print("subjects cleaned: trimmed, empty list -> null, max 5 items.")

subjects cleaned: trimmed, empty list -> null, max 5 items.


In [15]:
# Final columns (same schema: work_key, title, author_key, subjects)
df_cleaned = df_cleaned[["work_key", "title", "author_key", "subjects"]].copy()

print("Final schema:")
print(df_cleaned.dtypes)
print(f"\nFinal row count: {len(df_cleaned):,}")

Final schema:
work_key         str
title            str
author_key       str
subjects      object
dtype: object

Final row count: 40,688,902


## Step 4: Final Quality Check

In [16]:
print("=== Final Quality Check ===")
print(f"Rows with non-null title: {df_cleaned['title'].notna().sum():,}")
print(f"Rows with non-null author_key: {df_cleaned['author_key'].notna().sum():,}")
print(f"Rows with non-null subjects: {df_cleaned['subjects'].notna().sum():,}")
print("\nSample of cleaned data:")
df_cleaned.head(20)

=== Final Quality Check ===
Rows with non-null title: 40,688,902
Rows with non-null author_key: 38,458,493
Rows with non-null subjects: 19,854,712

Sample of cleaned data:


,work_key,title,author_key,subjects
0,/works/OL10000358W,Chacun et Tous,/authors/OL3965379A,NaN
1,/works/OL10000512W,PCEM intensif,/authors/OL3965552A,NaN
2,/works/OL1000092W,Highland princess,/authors/OL92939A,"[Historical romance, Historical fiction, Ficti..."
3,/works/OL1000101W,Wedding belles,/authors/OL92939A,"[Middle-aged women, Fiction, Female friendship..."
4,/works/OL10001112W,Connexions dangereuses,/authors/OL3966241A,NaN
5,/works/OL10001436W,The revolutionary war,/authors/OL3966610A,NaN
6,/works/OL10001472W,Contes du sud du Cameroun,/authors/OL3966638A,"[Tales, Legends, Folklore]"
7,/works/OL10001494W,The catalogue of allmost all the works done by...,/authors/OL3966666A,"[Exhibitions, Installations (Art)]"
8,/works/OL10001543W,"Vision d'escaflowne, tome 5",/authors/OL3966721A,NaN
9,/works/OL10001700W,"Cris du corps, nouvelles",/authors/OL3966880A,NaN


## Step 5: Save Cleaned Data

In [17]:
output_path = PROCESSED_DIR / 'works_cleaned.parquet'
df_cleaned.to_parquet(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved cleaned data to ../data/processed/works_cleaned.parquet
File size: 2320.48 MB


## Step 6: Generate Quality Report

In [19]:
report_path = REPORTS_DIR / 'data_quality_works_pandas.md'

report = f"""# Data Quality Report: Works Dataset (Pandas)

## Summary
- **Original row count**: {len(df):,}
- **Cleaned row count**: {len(df_cleaned):,}
- **Rows removed**: {len(df) - len(df_cleaned):,} ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)
- **Rows retained**: {len(df_cleaned)/len(df)*100:.2f}%

## Data Quality Metrics
- **Rows with non-null title**: {df_cleaned['title'].notna().sum():,}
- **Rows with non-null author_key**: {df_cleaned['author_key'].notna().sum():,}
- **Rows with non-null subjects**: {df_cleaned['subjects'].notna().sum():,}

## Schema (unchanged)
- work_key (String, required)
- title (String, required)
- author_key (String, nullable)
- subjects (List[String], nullable)

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Trimmed whitespace from title; removed rows with null/empty title
3. Cleaned author_key: strip; set invalid or empty to null
4. Cleaned subjects: trim items, empty list → null, limit to first 5 items
"""
report_path.write_text(report)
print(f"Quality report saved to {report_path}")
print("\n" + report_path.read_text())

Quality report saved to ../reports/data_quality_works_pandas.md

# Data Quality Report: Works Dataset (Pandas)

## Summary
- **Original row count**: 40,688,965
- **Cleaned row count**: 40,688,902
- **Rows removed**: 63 (0.00%)
- **Rows retained**: 100.00%

## Data Quality Metrics
- **Rows with non-null title**: 40,688,902
- **Rows with non-null author_key**: 38,458,493
- **Rows with non-null subjects**: 19,854,712

## Schema (unchanged)
- work_key (String, required)
- title (String, required)
- author_key (String, nullable)
- subjects (List[String], nullable)

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Trimmed whitespace from title; removed rows with null/empty title
3. Cleaned author_key: strip; set invalid or empty to null
4. Cleaned subjects: trim items, empty list → null, limit to first 5 items



## Summary

Done: load & inspect → quality assessment → work_key validation → title cleaning → author_key cleaning → subjects cleaning → save `works_cleaned.parquet` and quality report.